# Student Performance Prediction - Analysis Notebook
**Algorithm:** Random Forest Classifier (Scikit-learn)  |  **Data:** synthetic CSV dataset

Run `python generate_dataset.py` once before running this notebook. Open it from the project root folder
(or from `notebooks/`; the code below handles both).

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # move to project root so 'common.py' can be imported
sys.path.insert(0, os.getcwd())

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from common import *
from train_model import build_pipeline, RANDOM_STATE
sns.set_theme(style="whitegrid")

## 1. Load and understand the dataset

In [ ]:
raw = load_dataset()
print("Rows, columns:", raw.shape)
raw.head(10)

In [ ]:
print(raw.dtypes, "\n")
raw.describe().round(2)

In [ ]:
print("Missing values:\n", raw.isna().sum(), "\n")
print("Duplicate records:", raw.duplicated().sum())
print("\nTarget distribution:\n", raw[TARGET].value_counts())

## 2. Preprocessing
Duplicates are removed before the split. Missing feature values are filled **inside the pipeline** (median for numbers, most-frequent for categories) so the statistics come from training data only - this avoids data leakage.

In [ ]:
df = clean_dataset(raw)
print("Rows after removing duplicates:", len(df))
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 3. Exploratory Data Analysis

In [ ]:
palette = {"Poor": "#e76f51", "Average": "#f4a261", "Good": "#2a9d8f"}
plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x=TARGET, hue=TARGET, order=CLASS_ORDER, palette=palette, legend=False)
ax.bar_label(ax.containers[0]); plt.title("Distribution of Student Performance Categories")
plt.xlabel("Performance category"); plt.ylabel("Number of students"); plt.show()

In [ ]:
for col in ["Study_Hours_Per_Day", "Attendance_Percentage", "Previous_Exam_Marks", "Assignment_Scores"]:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x=TARGET, y=col, hue=TARGET, order=CLASS_ORDER, palette=palette, legend=False)
    plt.title(f"{LABELS[col]} vs Performance"); plt.xlabel("Performance category"); plt.ylabel(LABELS[col]); plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["Sleep_Hours"], bins=20, kde=True, color="#457b9d")
plt.title("Distribution of Sleep Hours"); plt.xlabel("Sleep hours per day"); plt.ylabel("Number of students"); plt.show()

corr = df[NUMERIC_FEATURES].copy()
corr["Performance (Poor=0, Avg=1, Good=2)"] = df[TARGET].map({c: i for i, c in enumerate(CLASS_ORDER)})
plt.figure(figsize=(9,7)); sns.heatmap(corr.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Heatmap"); plt.show()

In [ ]:
df.groupby(TARGET)[NUMERIC_FEATURES].mean().reindex(CLASS_ORDER).round(1)

**Observations:** students in the *Good* group have clearly higher median study hours, attendance, previous marks and assignment scores; the boxes overlap partly, so no single feature separates the groups perfectly. Sleep hours show only a weak relationship with performance.

## 4. Train the Random Forest

In [ ]:
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)
pipeline

## 5. Evaluate on the unseen test set

In [ ]:
y_pred = pipeline.predict(X_test)
print(f"Accuracy          : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (macro) : {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall (macro)    : {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-score (macro)  : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Training accuracy : {accuracy_score(y_train, pipeline.predict(X_train)):.4f}")
cv = cross_val_score(build_pipeline(), X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
print(f"5-fold CV accuracy: {cv.mean():.4f} +/- {cv.std():.4f}")
print("\n", classification_report(y_test, y_pred, labels=CLASS_ORDER))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
plt.figure(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
plt.title("Confusion Matrix (Test Data)"); plt.xlabel("Predicted category"); plt.ylabel("Actual category"); plt.show()

In [ ]:
names = [n.split("__", 1)[1] for n in pipeline.named_steps["preprocess"].get_feature_names_out()]
imp = pd.Series(pipeline.named_steps["model"].feature_importances_, index=names).sort_values()
imp.plot(kind="barh", color="#2a9d8f", figsize=(7,4.5)); plt.title("Feature Importance"); plt.xlabel("Importance"); plt.show()

## 6. Predict for one new student

In [ ]:
student = {"Study_Hours_Per_Day": 4.0, "Attendance_Percentage": 85, "Previous_Exam_Marks": 72,
           "Assignment_Scores": 75, "Internal_Assessment_Marks": 70, "Sleep_Hours": 7,
           "Extracurricular_Activity": "Yes", "Internet_Availability": "Yes"}
print("Input errors:", validate_input(student))
label, probs = predict_student(pipeline, student)
print("Predicted:", label); print({k: round(v, 3) for k, v in probs.items()})